# Phase 4 — Build a Prompt Eval Pipeline

When you change a prompt, how do you know it actually got better — not just better on one example you tried?

Answer: run it against a **dataset** of test cases, score each output with a **grader**, average the scores, then compare prompt versions by number.

## What you'll learn
1. The 4 pieces of an eval pipeline: **dataset · prompt · LLM · grader**
2. **Code-based grader** — programmatic check (does it parse as JSON?)
3. **Model-based grader** — a second LLM call that scores quality
4. The eval loop in action: change the prompt, watch the score move

Goal task for this notebook: "given a person description, extract structured JSON." We'll start with a deliberately bad prompt (score ~3-5), then improve it (score ~8-9) using techniques from Phase 3.

## Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

import json
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"
grader_model = "claude-haiku-4-5"   # cheaper for the grader — it doesn't need to be the smartest

## Step 1 — The dataset

5 hand-written test cases. Each case has:
- `task` — the input the prompt will see
- `expected_keys` — what the JSON output should contain (used by the model grader)

In real projects you'd have 50+ cases and might generate them with another LLM call. 5 is plenty for learning.

In [ ]:
dataset = [
    {
        "task": "Maria Lopez, 34, lives in Madrid, works as a data engineer at Acme Corp.",
        "expected_keys": ["name", "age", "city", "job", "employer"],
    },
    {
        "task": "Dr. Kenji Tanaka (52) is a cardiologist at Tokyo General Hospital.",
        "expected_keys": ["name", "age", "city", "job", "employer"],
    },
    {
        "task": "Aisha, age 28, software developer at Bumble, based out of Austin.",
        "expected_keys": ["name", "age", "city", "job", "employer"],
    },
    {
        "task": "Carlos (41) — sales manager, Bogota, Mercado Inc.",
        "expected_keys": ["name", "age", "city", "job", "employer"],
    },
    {
        "task": "Sophie Chen is 29 and works in Toronto as a UX designer for Shopify.",
        "expected_keys": ["name", "age", "city", "job", "employer"],
    },
]
print(f"Dataset has {len(dataset)} test cases.")

## Step 2 — `run_prompt`: produce an output for one test case

Takes a prompt template + one test case, returns the raw model output. The prompt has a `{task}` placeholder we interpolate.

In [ ]:
def run_prompt(prompt_template: str, test_case: dict) -> str:
    user_message = prompt_template.format(task=test_case["task"])
    response = client.messages.create(
        model=model,
        max_tokens=500,
        messages=[{"role": "user", "content": user_message}],
    )
    return response.content[0].text

## Step 3 — Grader #1: code-based (does it parse as JSON?)

The simplest grader: try `json.loads`. If it works, give 10; if not, 0. This catches the "Claude added markdown fences / preamble" problem from Phase 3.

In [ ]:
def grade_syntax(output: str) -> int:
    try:
        json.loads(output)
        return 10
    except json.JSONDecodeError:
        return 0

## Step 4 — Grader #2: model-based (does it answer the question?)

A second LLM call. The grader sees the original task + the model's output + a rubric, returns a JSON `{ score, reasoning }`. We use prefill + stop sequence (Phase 3 technique) to get clean JSON.

**Tip from the course:** ask for `strengths`, `weaknesses`, `reasoning`, *then* score. Models that are asked for the score alone tend to anchor on 5-7 by default.

In [ ]:
GRADER_PROMPT = '''You are grading an AI's output on a structured extraction task.

<original_task>
{task}
</original_task>

<ai_output>
{output}
</ai_output>

<rubric>
- The output should be a valid JSON object (no markdown, no commentary).
- It should contain these keys: {expected_keys}
- Values should be extracted accurately from the original task.
- Age should be a number, not a string.
</rubric>

Respond in JSON with: strengths (string), weaknesses (string), reasoning (string), score (integer 1-10).'''

def grade_quality(test_case: dict, output: str) -> dict:
    prompt = GRADER_PROMPT.format(
        task=test_case["task"],
        output=output,
        expected_keys=", ".join(test_case["expected_keys"]),
    )
    response = client.messages.create(
        model=grader_model,
        max_tokens=500,
        messages=[
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": "```json\n"}
        ],
        stop_sequences=["```"],
    )
    return json.loads(response.content[0].text)

## Step 5 — `run_eval`: tie it all together

For each test case: run the prompt → run both graders → record results. Average the scores.

In [ ]:
def run_eval(prompt_template: str, label: str = "") -> dict:
    print(f"\n=== Evaluating: {label} ===")
    results = []
    for i, case in enumerate(dataset, 1):
        output = run_prompt(prompt_template, case)
        syntax_score = grade_syntax(output)
        try:
            quality = grade_quality(case, output)
            quality_score = quality["score"]
            reasoning = quality["reasoning"]
        except Exception as e:
            quality_score = 0
            reasoning = f"grader failed: {e}"
        final = (syntax_score + quality_score) / 2
        results.append({
            "case": i,
            "syntax": syntax_score,
            "quality": quality_score,
            "final": final,
            "output": output[:80].replace("\n", " "),
            "reasoning": reasoning[:80],
        })
        print(f"  case {i}: syntax={syntax_score}  quality={quality_score}  final={final}")

    avg = sum(r["final"] for r in results) / len(results)
    print(f"  AVERAGE: {avg:.2f}")
    return {"label": label, "average": avg, "results": results}

## Step 6 — Run a deliberately bad prompt (baseline)

Watch how a vague prompt scores. The syntax grader should fail most cases — Claude will wrap the JSON in markdown fences and add commentary.

In [ ]:
prompt_v1 = "Please solve this task: {task}"
result_v1 = run_eval(prompt_v1, label="v1 — vague")

Probably scored 2-5 on average. Let's see the actual outputs to confirm why.

In [ ]:
for r in result_v1["results"]:
    print(f"  case {r['case']} → {r['output']!r}")

## Step 7 — Apply prompt engineering, re-run

Now we apply techniques from the course: clear action verb, explicit format, XML tag for the input, exact key list. No magic — just spelling out what we want.

In [ ]:
prompt_v2 = '''Extract structured biographical data from the input below and respond with ONLY a JSON object.

Rules:
- Output must be a single valid JSON object, no markdown fences, no explanatory text.
- Required keys: name (string), age (integer), city (string), job (string), employer (string).
- If a field is unclear, make your best inference from the input.

<input>
{task}
</input>'''

result_v2 = run_eval(prompt_v2, label="v2 — clear + specific + XML")

## Step 8 — Compare versions side by side

In [ ]:
print(f"  v1 (vague):           {result_v1['average']:.2f}")
print(f"  v2 (clear+specific):  {result_v2['average']:.2f}")
delta = result_v2['average'] - result_v1['average']
print(f"  improvement:          {delta:+.2f}")

**That's the eval loop in action.** Before, you'd squint at outputs and go "feels better." Now you have a number. Change a prompt → re-run → see if the number moved.

## Mini-exercise

1. **Push v2 to ~10.** What's left to improve? Try adding a one-shot example (a sample input + ideal JSON) to `prompt_v2`. Re-run. Does it close the gap?
2. **Stress-test the grader.** Manually write a JSON output that *looks* right but has the age as a string (`"age": "34"` instead of `"age": 34`). Pass it through `grade_quality` directly. Does the grader catch the violation? If not, that's a hint to make the rubric stricter.
3. **Add a corner case.** Add a 6th test case with ambiguous input (e.g. *"Pat Smith, works at Globex"* — no age, no city). Watch how v2 handles missing fields. Adjust the prompt's "if unclear" rule based on what you see.

Reply with the v2 average score you got and one observation. Next: **Phase 5 — tool use**, where we build the reminder bot end-to-end with the full multi-tool loop.